In [4]:
import zipfile
import polars as pl
import io

zip_path = r'C:\Users\bened\Downloads\Archive (1).zip'

file_names = [
    'sample_user_note_traj',
    'sample_user_rating_traj',
    'sample_user_request_traj',
]

dataframes = {}
with zipfile.ZipFile(zip_path, 'r') as z:
    for name in file_names:
        matches = [f for f in z.namelist() if name in f and f.endswith('.parquet')]
        if not matches:
            print(f'Could not find {name}.parquet in zip')
            continue
        with z.open(matches[0]) as f:
            dataframes[name] = pl.read_parquet(io.BytesIO(f.read()))

for name, df in dataframes.items():
    print(f'\n{name} columns:')
    print(df.columns)



sample_user_note_traj columns:
['noteAuthorParticipantId', 'userMonth', 'sportsCount', 'diaries_&_daily_lifeCount', 'business_&_entrepreneursCount', 'science_&_technologyCount', 'news_&_social_concernCount', 'otherCount', 'calendarMonth', 'notesCreated', 'hitRate', 'hits', 'avgNoteFactor', 'avgNoteIntercept', 'topicsTargeted', 'avgRatingsEarned']

sample_user_rating_traj columns:
['raterParticipantId', 'userMonth', 'sportsRatedCount', 'diaries_&_daily_lifeRatedCount', 'business_&_entrepreneursRatedCount', 'science_&_technologyRatedCount', 'news_&_social_concernRatedCount', 'otherRatedCount', 'calendarMonth', 'notesRated', 'avgHelpfulFactor', 'avgNotHelpfulFactor', 'avgHelpfulIntercept', 'avgNotHelpfulIntercept', 'correctHelpfuls', 'correctNotHelpfuls', 'posFactorRatedHelpful', 'posFactorRatedNotHelpful', 'negFactorRatedHelpful', 'negFactorRatedNotHelpful', 'pctCorrectPosFactorHelpful', 'pctCorrectPosFactorNotHelpful', 'pctCorrectNegFactorHelpful', 'pctCorrectNegFactorNotHelpful', 'pct

In [5]:
join_keys = ["userId", "userMonth", "calendarMonth"]

notes = dataframes["sample_user_note_traj"].rename({"noteAuthorParticipantId": "userId"})
ratings = dataframes["sample_user_rating_traj"].rename({"raterParticipantId": "userId"})
requests = dataframes["sample_user_request_traj"].rename({"requesterParticipantId": "userId"})

master = (
    notes
    .join(ratings, on=join_keys, how="full", coalesce=True)
    .join(requests, on=join_keys, how="full", coalesce=True)
    .sort(["userId", "userMonth"])
)

print(master.shape)
print(master.columns)
master.head()


(92281, 62)
['userId', 'userMonth', 'sportsCount', 'diaries_&_daily_lifeCount', 'business_&_entrepreneursCount', 'science_&_technologyCount', 'news_&_social_concernCount', 'otherCount', 'calendarMonth', 'notesCreated', 'hitRate', 'hits', 'avgNoteFactor', 'avgNoteIntercept', 'topicsTargeted', 'avgRatingsEarned', 'sportsRatedCount', 'diaries_&_daily_lifeRatedCount', 'business_&_entrepreneursRatedCount', 'science_&_technologyRatedCount', 'news_&_social_concernRatedCount', 'otherRatedCount', 'notesRated', 'avgHelpfulFactor', 'avgNotHelpfulFactor', 'avgHelpfulIntercept', 'avgNotHelpfulIntercept', 'correctHelpfuls', 'correctNotHelpfuls', 'posFactorRatedHelpful', 'posFactorRatedNotHelpful', 'negFactorRatedHelpful', 'negFactorRatedNotHelpful', 'pctCorrectPosFactorHelpful', 'pctCorrectPosFactorNotHelpful', 'pctCorrectNegFactorHelpful', 'pctCorrectNegFactorNotHelpful', 'pctHelpfulRatingsCorrect', 'pctNotHelpfulRatingsCorrect', 'uniqueDaysRated', 'avgPostsRatedPerDay', 'uniqueTopicsRated', 'antiD

userId,userMonth,sportsCount,diaries_&_daily_lifeCount,business_&_entrepreneursCount,science_&_technologyCount,news_&_social_concernCount,otherCount,calendarMonth,notesCreated,hitRate,hits,avgNoteFactor,avgNoteIntercept,topicsTargeted,avgRatingsEarned,sportsRatedCount,diaries_&_daily_lifeRatedCount,business_&_entrepreneursRatedCount,science_&_technologyRatedCount,news_&_social_concernRatedCount,otherRatedCount,notesRated,avgHelpfulFactor,avgNotHelpfulFactor,avgHelpfulIntercept,avgNotHelpfulIntercept,correctHelpfuls,correctNotHelpfuls,posFactorRatedHelpful,posFactorRatedNotHelpful,negFactorRatedHelpful,negFactorRatedNotHelpful,pctCorrectPosFactorHelpful,pctCorrectPosFactorNotHelpful,pctCorrectNegFactorHelpful,pctCorrectNegFactorNotHelpful,pctHelpfulRatingsCorrect,pctNotHelpfulRatingsCorrect,uniqueDaysRated,avgPostsRatedPerDay,uniqueTopicsRated,antiDemNNRatings,proDemNNRatings,proDemNNNRatings,antiDemNNNRatings,antiRepNNRatings,proRepNNRatings,proRepNNNRatings,antiRepNNNRatings,overallAccuracy,helpfulNotHelpfulFactorDiff,helpfulNotHelpfulInterceptDiff,proDemRatings,antiDemRatings,proRepRatings,antiRepRatings,requestsMade,numRequestsResultingInCrh,numRequestsResultingInNote,pctRequestResultedInNote,pctRequestResultedInCrh
str,i32,u32,u32,u32,u32,u32,u32,str,u32,f64,u32,f64,f64,u32,f64,u32,u32,u32,u32,u32,u32,u32,f64,f64,f64,f64,u32,u32,u32,u32,u32,u32,f64,f64,f64,f64,f64,f64,u32,f64,u32,u32,u32,u32,u32,u32,u32,u32,u32,f64,f64,f64,u32,u32,u32,u32,u32,u32,u32,f64,f64
"""000AE77955227AE8D52CB70BA3FB64…",0,null,null,null,null,null,null,"""2025-12""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1,0,0,0.0,0.0
"""000C7AC0F2AE15FB6C478257825D41…",0,null,null,null,null,null,null,"""2025-11""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1,0,0,0.0,0.0
"""0014F9BA334FB10689F9D089BF4791…",0,null,null,null,null,null,null,"""2025-08""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1,0,0,0.0,0.0
"""00167DE211B4D41B0E054EFD79A977…",0,null,null,null,null,null,null,"""2025-11""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1,0,0,0.0,0.0
"""00167DE211B4D41B0E054EFD79A977…",1,null,null,null,null,null,null,"""2025-12""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1,0,0,0.0,0.0


In [7]:
# Fill nulls with 0 for activity columns (null = no activity of that type that month)
notes_col = pl.col("notesCreated").fill_null(0)
rated_col = pl.col("notesRated").fill_null(0)
reqs_col  = pl.col("requestsMade").fill_null(0)

classification = (
    pl.when(notes_col == 1).then(pl.lit("single_note_writer"))
    .when(notes_col.is_between(2, 9)).then(pl.lit("single_digit_writer"))
    .when(notes_col >= 10).then(pl.lit("double_digit_writer"))
    .when((notes_col == 0) & (rated_col == 1)).then(pl.lit("single_note_rater"))
    .when((notes_col == 0) & rated_col.is_between(2, 9)).then(pl.lit("single_digit_rater"))
    .when((notes_col == 0) & (rated_col >= 10)).then(pl.lit("double_digit_rater"))
    .when((notes_col == 0) & (rated_col == 0) & (reqs_col == 1)).then(pl.lit("single_post_requestor"))
    .when((notes_col == 0) & (rated_col == 0) & reqs_col.is_between(2, 9)).then(pl.lit("single_digit_requestor"))
    .when((notes_col == 0) & (rated_col == 0) & (reqs_col >= 10)).then(pl.lit("double_digit_requestor"))
    .otherwise(pl.lit("not_active"))
)

# multiple_categories: user was active in more than one of notes / ratings / requests
multi = (
    ((notes_col >= 1).cast(pl.Int8) +
     (rated_col >= 1).cast(pl.Int8) +
     (reqs_col  >= 1).cast(pl.Int8)) > 1
)

master_classified = master.with_columns([
    classification.alias("classification"),
    multi.alias("multiple_categories"),
])

print(master_classified["classification"].value_counts(sort=True))
master_classified

shape: (9, 2)
┌────────────────────────┬───────┐
│ classification         ┆ count │
│ ---                    ┆ ---   │
│ str                    ┆ u32   │
╞════════════════════════╪═══════╡
│ single_digit_rater     ┆ 29655 │
│ single_note_rater      ┆ 21890 │
│ single_post_requestor  ┆ 17126 │
│ double_digit_rater     ┆ 13646 │
│ single_digit_requestor ┆ 5358  │
│ single_note_writer     ┆ 2648  │
│ single_digit_writer    ┆ 1459  │
│ double_digit_requestor ┆ 415   │
│ double_digit_writer    ┆ 84    │
└────────────────────────┴───────┘


userId,userMonth,sportsCount,diaries_&_daily_lifeCount,business_&_entrepreneursCount,science_&_technologyCount,news_&_social_concernCount,otherCount,calendarMonth,notesCreated,hitRate,hits,avgNoteFactor,avgNoteIntercept,topicsTargeted,avgRatingsEarned,sportsRatedCount,diaries_&_daily_lifeRatedCount,business_&_entrepreneursRatedCount,science_&_technologyRatedCount,news_&_social_concernRatedCount,otherRatedCount,notesRated,avgHelpfulFactor,avgNotHelpfulFactor,avgHelpfulIntercept,avgNotHelpfulIntercept,correctHelpfuls,correctNotHelpfuls,posFactorRatedHelpful,posFactorRatedNotHelpful,negFactorRatedHelpful,negFactorRatedNotHelpful,pctCorrectPosFactorHelpful,pctCorrectPosFactorNotHelpful,pctCorrectNegFactorHelpful,pctCorrectNegFactorNotHelpful,pctHelpfulRatingsCorrect,pctNotHelpfulRatingsCorrect,uniqueDaysRated,avgPostsRatedPerDay,uniqueTopicsRated,antiDemNNRatings,proDemNNRatings,proDemNNNRatings,antiDemNNNRatings,antiRepNNRatings,proRepNNRatings,proRepNNNRatings,antiRepNNNRatings,overallAccuracy,helpfulNotHelpfulFactorDiff,helpfulNotHelpfulInterceptDiff,proDemRatings,antiDemRatings,proRepRatings,antiRepRatings,requestsMade,numRequestsResultingInCrh,numRequestsResultingInNote,pctRequestResultedInNote,pctRequestResultedInCrh,classification,multiple_categories
str,i32,u32,u32,u32,u32,u32,u32,str,u32,f64,u32,f64,f64,u32,f64,u32,u32,u32,u32,u32,u32,u32,f64,f64,f64,f64,u32,u32,u32,u32,u32,u32,f64,f64,f64,f64,f64,f64,u32,f64,u32,u32,u32,u32,u32,u32,u32,u32,u32,f64,f64,f64,u32,u32,u32,u32,u32,u32,u32,f64,f64,str,bool
"""000AE77955227AE8D52CB70BA3FB64…",0,null,null,null,null,null,null,"""2025-12""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1,0,0,0.0,0.0,"""single_post_requestor""",false
"""000C7AC0F2AE15FB6C478257825D41…",0,null,null,null,null,null,null,"""2025-11""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1,0,0,0.0,0.0,"""single_post_requestor""",false
"""0014F9BA334FB10689F9D089BF4791…",0,null,null,null,null,null,null,"""2025-08""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1,0,0,0.0,0.0,"""single_post_requestor""",false
"""00167DE211B4D41B0E054EFD79A977…",0,null,null,null,null,null,null,"""2025-11""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1,0,0,0.0,0.0,"""single_post_requestor""",false
"""00167DE211B4D41B0E054EFD79A977…",1,null,null,null,null,null,null,"""2025-12""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1,0,0,0.0,0.0,"""single_post_requestor""",false
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""FFF473DD905D41503CF2EF95C6B3AF…",19,null,null,null,null,null,null,"""2025-07""",null,null,null,null,null,null,null,0,0,0,0,1,0,1,0.249378,null,0.472519,null,1,0,1,0,0,0,1.0,null,null,null,1.0,null,1,1.0,1,0,0,0,0,0,0,0,0,1.0,null,null,0,0,0,0,null,null,null,null,null,"""single_note_rater""",false
"""FFF473DD905D41503CF2EF95C6B3AF…",20,null,null,null,null,null,null,"""2025-08""",null,null,null,null,null,null,null,0,0,0,0,1,0,1,0.26023,null,0.461019,null,1,0,1,0,0,0,1.0,null,null,null,